# 第 12 天 · 工具的健壮性

> 📓 **这是第 12 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/12)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q openai pydantic python-dotenv
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
"""第 12 天 · 工具的健壮性：参数校验 + 错误回传 + 超时 + 防死循环
没有 Key 也能跑（假模型故意犯错，再根据错误信息自我纠正）；填了 LLM_API_KEY 就调用真模型。
要点：① 工具永不抛异常，错误统一变成 JSON 告诉模型 ② 错误信息要写清"错在哪、怎么改"
      ③ 每个工具有超时 ④ 最大轮数 + 重复调用检测，防止死循环"""
import ast
import asyncio
import json
import operator
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError, field_validator

load_dotenv()


# ---------- 1. 工具参数：用 pydantic 约束（类比 @NotBlank @Pattern @Max） ----------
FAKE_WEATHER = {"北京": {"condition": "晴", "temp_c": 12}, "上海": {"condition": "小雨", "temp_c": 18}}


class WeatherArgs(BaseModel):
    city: str = Field(min_length=1, max_length=10, description="城市中文名，如 北京、上海；不要带'市'字")

    @field_validator("city")
    @classmethod
    def strip_city(cls, v: str) -> str:
        return v.strip().removesuffix("市")  # 能宽容的就宽容：'北京市' → '北京'


async def get_weather(city: str) -> dict:
    await asyncio.sleep(0.1)
    if city not in FAKE_WEATHER:
        # 业务错误：告诉模型"支持哪些"，它才能改对
        raise ToolError(f"没有 {city} 的天气数据", hint=f"只支持：{', '.join(FAKE_WEATHER)}")
    return {"city": city, **FAKE_WEATHER[city]}


class CalcArgs(BaseModel):
    expression: str = Field(max_length=100, description="只含数字和 + - * / ** ( ) 的算术表达式，如 (18-12)*3")


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        left, right = _eval(node.left), _eval(node.right)
        if isinstance(node.op, ast.Pow) and (abs(right) > 100 or abs(left) > 10_000):
            raise ToolError("指数太大，拒绝计算", hint="底数绝对值 ≤ 10000、指数绝对值 ≤ 100")  # 防 9**9**9
        return _OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ToolError("表达式里有不允许的内容", hint="只能用数字和 + - * / ** ( )")


async def calculator(expression: str) -> dict:
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        raise ToolError(f"表达式语法错误：{expression!r}", hint="检查括号是否配对，例如 (18-12)*3")
    return {"expression": expression, "result": _eval(tree.body)}


class SlowArgs(BaseModel):
    query: str = Field(description="要查询的内容")


async def slow_search(query: str) -> dict:
    await asyncio.sleep(10)  # 模拟一个卡住的下游接口
    return {"query": query, "items": []}


# ---------- 2. 统一的错误类型和执行器（类比 @ControllerAdvice 全局异常处理） ----------
class ToolError(Exception):
    """可预期的业务错误：带一句给模型看的修改建议"""
    def __init__(self, message: str, hint: str = ""):
        super().__init__(message)
        self.hint = hint


REGISTRY = {
    "get_weather": (WeatherArgs, get_weather, "查询城市当前天气和气温。", 2.0),
    "calculator": (CalcArgs, calculator, "精确计算算术表达式，任何算数都用它。", 1.0),
    "slow_search": (SlowArgs, slow_search, "搜索网页（可能很慢）。", 1.0),
}  # 每项：(参数模型, 函数, 说明, 超时秒数)

TOOLS = [{"type": "function",
          "function": {"name": n, "description": d, "parameters": m.model_json_schema()}}
         for n, (m, _, d, _) in REGISTRY.items()]


def err(kind: str, message: str, hint: str = "", retryable: bool = True) -> str:
    return json.dumps({"ok": False, "error_type": kind, "message": message,
                       "hint": hint, "retryable": retryable}, ensure_ascii=False)


async def run_tool(name: str, arguments: str) -> str:
    """无论发生什么都返回字符串，绝不把异常抛给 Agent 循环"""
    if name not in REGISTRY:
        return err("unknown_tool", f"没有叫 {name} 的工具", hint=f"可用工具：{', '.join(REGISTRY)}")
    args_model, func, _, timeout = REGISTRY[name]
    try:
        args = args_model.model_validate_json(arguments or "{}")       # ① JSON 解析 + 参数校验
    except ValidationError as e:
        problems = "; ".join(f"{'.'.join(map(str, x['loc'])) or '参数'}: {x['msg']}" for x in e.errors())
        return err("invalid_arguments", problems, hint="按 tools 里的 JSON Schema 重新传参")
    try:
        result = await asyncio.wait_for(func(**args.model_dump()), timeout)  # ② 超时
        return json.dumps({"ok": True, "data": result}, ensure_ascii=False)
    except ToolError as e:                                              # ③ 业务错误：可改正
        return err("tool_error", str(e), hint=e.hint)
    except asyncio.TimeoutError:                                        # ④ 超时：别再试了
        return err("timeout", f"{name} 超过 {timeout}s 没响应", hint="换别的办法或直接告诉用户", retryable=False)
    except Exception as e:                                              # ⑤ 兜底：程序 bug
        return err("internal_error", f"{type(e).__name__}: {e}", retryable=False)


# ---------- 3. 模型 ----------
def _call(cid, name, args):
    raw = args if isinstance(args, str) else json.dumps(args, ensure_ascii=False)
    return {"id": cid, "type": "function", "function": {"name": name, "arguments": raw}}


async def fake_llm(messages, tools):
    """剧本：先犯 3 种错，看到错误信息后逐个改正，最后作答"""
    await asyncio.sleep(0.05)
    done = {m["tool_call_id"]: json.loads(m["content"]) for m in messages if m["role"] == "tool"}
    if not done:  # 第 1 轮：参数 JSON 坏了 + 城市不支持 + 工具名写错
        return {"role": "assistant", "content": None, "tool_calls": [
            _call("c1", "get_weather", '{"city": "北京"'),          # 少了 }
            _call("c2", "get_weather", {"city": "广州"}),
            _call("c3", "get_wether", {"city": "上海"}),
        ]}
    if "c4" not in done:  # 第 2 轮：根据 hint 改正
        return {"role": "assistant", "content": None, "tool_calls": [
            _call("c4", "get_weather", {"city": "北京市"}),
            _call("c5", "get_weather", {"city": "上海"}),
            _call("c6", "calculator", {"expression": "9**9**9"}),  # 又犯一个错
        ]}
    if "c7" not in done:
        bj, sh = done["c4"]["data"]["temp_c"], done["c5"]["data"]["temp_c"]
        return {"role": "assistant", "content": None,
                "tool_calls": [_call("c7", "calculator", {"expression": f"{sh}-{bj}"})]}
    return {"role": "assistant", "content":
            f"广州暂时查不到。上海 {done['c5']['data']['temp_c']}°C，北京 {done['c4']['data']['temp_c']}°C，"
            f"上海高 {done['c7']['data']['result']}°C。"}


async def stubborn_llm(messages, tools):
    """反面角色：不管返回什么，永远重复同一个调用"""
    return {"role": "assistant", "content": None,
            "tool_calls": [_call(f"s{len(messages)}", "slow_search", {"query": "广州天气"})]}


async def real_llm(messages, tools):
    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=os.environ["LLM_API_KEY"],
                         base_url=os.getenv("LLM_BASE_URL", "https://api.deepseek.com"))
    resp = await client.chat.completions.create(
        model=os.getenv("LLM_MODEL", "deepseek-chat"), messages=messages, tools=tools, temperature=0)
    msg = resp.choices[0].message
    out = {"role": "assistant", "content": msg.content}
    if msg.tool_calls:
        out["tool_calls"] = [{"id": c.id, "type": "function",
                              "function": {"name": c.function.name, "arguments": c.function.arguments}}
                             for c in msg.tool_calls]
    return out


# ---------- 4. Agent 循环：最大轮数 + 重复调用检测 ----------
MAX_ROUNDS = 5
MAX_SAME_CALL = 2  # 同一个工具 + 同样参数最多执行 2 次


async def ask(question: str, llm) -> str:
    messages = [
        {"role": "system", "content": "你是助手。天气和计算必须调用工具。工具返回 ok=false 时，"
                                      "读 hint 改正参数再试；retryable=false 就别再试，直接如实告诉用户。"},
        {"role": "user", "content": question},
    ]
    seen: dict[tuple, int] = {}
    print(f"用户：{question}")
    for round_no in range(1, MAX_ROUNDS + 1):
        reply = await llm(messages, TOOLS)
        if not reply.get("tool_calls"):
            return reply["content"]
        messages.append(reply)
        print(f"\n第 {round_no} 轮：")

        async def handle(c):
            key = (c["function"]["name"], c["function"]["arguments"])
            seen[key] = seen.get(key, 0) + 1
            if seen[key] > MAX_SAME_CALL:
                return err("repeated_call", "同样的调用已经执行过，结果不会变", retryable=False)
            return await run_tool(*key)

        results = await asyncio.gather(*(handle(c) for c in reply["tool_calls"]))
        for c, result in zip(reply["tool_calls"], results):
            print(f"  → {c['function']['name']}({c['function']['arguments']})")
            print(f"  ← {result}")
            messages.append({"role": "tool", "tool_call_id": c["id"], "content": result})
    return f"⚠️ 超过 {MAX_ROUNDS} 轮还没得出答案，强制停止（真实项目里这里要打告警日志）。"


async def main():
    llm = real_llm if os.getenv("LLM_API_KEY") else fake_llm
    print("使用", "真模型" if llm is real_llm else "假模型（没填 Key）", "\n")
    answer = await ask("北京、上海、广州现在多少度？北京和上海差几度？", llm)
    print("\n最终回答：", answer)

    print("\n===== 反面教材：一直重复调用的模型 =====")
    print("\n最终回答：", await ask("广州天气怎么样？", stubborn_llm))


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await